# torch.optim 完整系统讲解笔记
## 目录
0. 优化器全景总览：torch.optim 支持的优化方法一览
1. 模块定位与核心作用 + 最简入门示例
2. 底层核心存储结构：param_groups / state（打印查看实例）
3. 通用核心API全解析（逐API配套代码演示）
4. 主流优化器原理、公式、对比（三种优化器实例化对照）
5. 配套工具：lr_scheduler 学习率调度器（4种调度器实操代码）
6. 高级用法：分层差异化参数训练（多组参数完整示例）
7. 完整标准训练流程代码模板（可直接运行）
8. 断点保存/加载优化器完整实例
9. 高频踩坑问题与避坑指南（错误示例+正确写法对照）
10. 附录：SGD每次迭代使用的样本数量（batch_size详解与设置方法）

# 第0章：优化器全景总览

## 0.0 核心理念：所有优化器的本质是"梯度消费者"

**所有优化器都使用梯度，只不过使用方式不同。**

```
                   所有优化器的输入
                          ↓
                    loss.backward()
                          ↓
                   参数.grad = g_t
                          ↓
                  optimizer.step()
                    ┌─────┴─────┐                      
            直接使用   平滑处理  自适应缩放...        ← 不同优化器在这里做不同的事情
            (SGD)   (动量SGD) (Adam/AdamW)...      
                  θ_new = θ_old - Δθ
                   └────────────┘
                         
                   
```

**关键职责划分：**
- `loss.backward()`：计算梯度，存入 `param.grad`
- `optimizer.step()`：读取 `param.grad`，执行参数更新 

| 策略类型 | 代表优化器 | 核心思想 | step() 具体操作 |
|---------|-----------|----------|----------------|
| **直接使用** | SGD（无动量） | `θ = θ - lr × g_t` | 直接读取 `.grad` 做减法 |
| **平滑处理** | SGD + momentum | 用历史梯度指数平均平滑震荡：`θ = θ - lr × m_t` | 先更新动量缓存，再更新参数 |
| **自适应缩放** | Adam/AdamW | 每个参数独立学习率：一阶动量+二阶动量自适应 | 先更新一阶/二阶动量，修正偏差，再更新参数 |

## 0.1 官方内置优化器完整清单

PyTorch `torch.optim` 模块目前支持以下优化算法（按类别分组）：

| 类别 | 优化器名称 | 核心特点 | 适用场景 |
|------|-----------|----------|----------|
| **基础SGD系列** | `SGD` | 带动量/无动量随机梯度下降，支持Nesterov加速 | 小数据集、泛化要求高、调参经验丰富 |
| **自适应学习率** | `Adam` | 一阶+二阶动量自适应，含偏差修正 | 快速收敛、NLP/Transformer、默认首选之一 |
| | `AdamW` | Adam解耦版，权重衰减独立后置 | **工业标准推荐**，泛化优于Adam |
| | `Adamax` | Adam无穷范数版本，对稀疏梯度鲁棒 | 稀疏梯度场景 |
| | `Adadelta` | 无学习率超参数，自适应梯度平方 | 不需要手动调lr的场景 |
| | `Adagrad` | 自适应学习率，参数频率不同步长不同 | 稀疏特征/推荐系统 |
| **RMSProp系列** | `RMSprop` | 梯度平方移动平均，自适应缩放 | RNN/强化学习、非平稳目标 |
| **NAdam系列** | `NAdam` | Adam + Nesterov动量 | 加速收敛，CV/NLP均可用 |
| **RAdam系列** | `RAdam` | Adam + 动态整流，解决早期方差不稳定 | 小batch/高噪声场景 |
| **稀疏优化** | `SparseAdam` | 专为稀疏梯度优化 | 词嵌入Embedding训练 |
| **LBFGS** | `LBFGS` | 拟牛顿法，二阶近似 | 小规模参数、精确收敛 |

## 0.2 最常用优化器速查（⚡ 推荐）

| 优化器 | 一句话总结 | 推荐场景 |
|--------|-----------|----------|
| **AdamW** | **当前工业界通用最优解**，收敛快+泛化强+正则稳定 | 所有深度学习任务的默认首选 |
| **SGD + momentum** | 经典泛化王者，需要精细调参 | 小数据集、CV传统架构、追求极致泛化 |
| **Adam** | 历史常用版本（但建议被AdamW替代） | 快速原型验证 |
| **RMSprop** | 适合循环神经网络和强化学习 | RNN/LSTM/Policy Gradient |

## 0.3 标准导入方式

In [ ]:
import torch.optim as optim
from torch.optim import Optimizer

print("PyTorch torch.optim 所有内置优化器类：")

optimizer_classes = [
    name for name in dir(optim) 
    if isinstance(getattr(optim, name), type)
    and issubclass(getattr(optim, name), Optimizer)
    and name not in ['Optimizer', 'LRScheduler']
]

print(", ".join(optimizer_classes))
print(f"\n共计 {len(optimizer_classes)} 种优化器")

# 一、torch.optim 模块定位与核心作用
## 1.1 本质定义
`torch.optim` 是PyTorch官方内置**梯度优化工具包**，专门负责神经网络参数更新：
- 反向传播算出梯度 `loss.backward()` 后，由优化器按照梯度下降算法修正权重、偏置；
- 封装了全部主流梯度优化算法，统一对外提供标准化接口；
- 管理参数组、迭代历史缓存（动量、二阶方差）、动态超参修改。
## 1.2 训练链路中的位置
数据前向传播计算损失 → 反向传播求梯度 → torch.optim 更新模型参数 → 迭代循环
## 1.3 导入方式

In [ ]:
import torch.optim as optim
# 或单独导入优化器
from torch.optim import SGD, Adam, AdamW

## 1.4 最简可运行入门示例

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
torch.manual_seed(42)
# 1. 最简单单层模型
net = nn.Linear(2, 1)
# 2. 初始化优化器：绑定模型所有参数
optimizer = optim.AdamW(net.parameters(), lr=0.01)
# 模拟数据
x = torch.tensor([[1.0, 2.0]])
y_true = torch.tensor([[3.0]])
loss_func = nn.MSELoss()

# 一次完整更新流程
optimizer.zero_grad()  # 清空梯度
y_pred = net(x)       # 前向
loss = loss_func(y_pred, y_true)
loss.backward()       # 反向求梯度 → 生成 .grad
optimizer.step()      # 优化器读取 .grad 并更新参数
print("更新后的权重：", net.weight)
print("\n💡 loss.backward() 计算梯度，optimizer.step() 执行更新")

细致地看寻优的步骤：

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
torch.manual_seed(42)

net = nn.Linear(2, 1)

# 🔍 显式列出需要优化的张量
weight = net.weight
bias = net.bias

print("需要优化的张量：")
print(f"  weight: requires_grad={weight.requires_grad}")
print(f"  bias:   requires_grad={bias.requires_grad}")
print()

# 方式1：传入参数列表（最明确）
optimizer = optim.AdamW([weight, bias], lr=0.01)
# 等价于：optimizer = optim.AdamW(net.parameters(), lr=0.01)

x = torch.tensor([[1.0, 2.0]])
y_true = torch.tensor([[3.0]])
loss_func = nn.MSELoss()

# 保存更新前的参数
old_weight = weight.data.clone()
old_bias = bias.data.clone()

print("=== 更新前 ===")
print(f"weight: {weight.data}")
print(f"bias:   {bias.data}")
print()

optimizer.zero_grad()
y_pred = net(x)    # 开始构建计算图
loss = loss_func(y_pred, y_true)
loss.backward()
optimizer.step()

print("=== 更新后 ===")
print(f"weight: {weight.data}")
print(f"bias:   {bias.data}")
print()

print(f"weight 变化量: {weight.data - old_weight}")
print(f"bias 变化量:   {bias.data - old_bias}")
print(f"\n✅ weight 是否更新: {torch.allclose(weight.data, old_weight) == False}")
print(f"✅ bias 是否更新:   {torch.allclose(bias.data, old_bias) == False}")
print("\n💡 优化器明确更新了 weight 和 bias 这两个张量！")

## 1.5 疑点：optimizer 和 loss 表面分离，二者如何建立联系？

### 🎭 "明修栈道，暗度陈仓"的背地交易

表面上看，`optimizer` 和 `loss` 毫无关系：
- `optimizer` 只认识 `net.parameters()`
- `loss` 只负责计算损失值

但背地里，它们通过 `param.grad` 完成"秘密交易"：

```
表面上看：
    optimizer ←→ net          (光明正大)
    loss ←→ net               (光明正大)
    optimizer ✗ loss          (看似毫无关系)

实际上：
    loss.backward()
         ↓
    把"梯度的情报"偷偷写入 → net.weight.grad
         ↓
    optimizer.step()
         ↓
    从 net.weight.grad 读取"情报"
         ↓
    修改 net.weight 执行更新
```

### 核心结论
优化器 `optimizer` 不会直接读取loss变量，二者唯一桥梁是**参数梯度 `.grad`**：
1. `loss.backward()` 自动链式求导，给模型权重挂载梯度 `w.grad`；
2. `optimizer.step()` 读取参数上的梯度，沿梯度反方向修正权重，下一轮前向计算loss自然减小。

**一句话总结：`loss.backward()`负责算梯度（存入.grad），`optimizer.step()`负责用梯度更新参数（修改权重值）。**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)
net = nn.Linear(2, 1)
# 纯基础SGD，无动量、无正则
optimizer = optim.SGD(net.parameters(), lr=0.01, momentum=0, weight_decay=0)

x = torch.tensor([[1.0, 2.0]])
y_true = torch.tensor([[3.0]])
loss_func = nn.MSELoss()

# 保存更新前参数
w_old = net.weight.clone()
b_old = net.bias.clone()

optimizer.zero_grad()
y_pred = net(x)
loss = loss_func(y_pred, y_true)
loss.backward()  # ← 计算梯度，存入 .grad

# 提取梯度（所有优化器都从这里读取数据）
grad_w = net.weight.grad
grad_b = net.bias.grad

optimizer.step()  # ← 读取 .grad，执行参数更新
w_new = net.weight.clone()
b_new = net.bias.clone()

# 统一整合算式打印
print("===== 权重完整更新算式 w_new = w_old - lr * grad_w =====")
print(f"w_old = {w_old}")
print(f"lr = 0.01")
print(f"grad_w = {grad_w}")
print(f"lr * grad_w = {0.01 * grad_w}")
print(f"w_new = w_old - lr * grad_w = {w_old - 0.01 * grad_w}")
print(f"实际更新后w_new = {w_new}")
print("\n===== 偏置完整更新算式 b_new = b_old - lr * grad_b =====")
print(f"b_old = {b_old.item():.6f}")
print(f"grad_b = {grad_b.item():.6f}")
print(f"lr * grad_b = {0.01 * grad_b.item():.6f}")
print(f"b_new = b_old - lr * grad_b = {(b_old - 0.01 * grad_b).item():.6f}")
print(f"实际更新后b_new = {b_new.item():.6f}")
print("\n💡 关键洞察：")
print("  - loss.backward() → 计算梯度，写入 param.grad")
print("  - optimizer.step() → 读取 param.grad，执行参数更新")
print("  - 这就是 optimizer 和 loss 的'背地交易'！")

### 🤔 为什么不是"显式函数传递"？

如果设计成显式传递，代码会变成这样：

```python
# ❌ 假想的显式设计（不够灵活）
grad = loss.backward()           # 返回梯度
optimizer.step(grad)             # 显式传入梯度
```

**核心问题：效率低、不省心！**

| 问题 | 说明 |
|------|------|
| **参数数量巨大** | 复杂网络有数百万参数，需要传递整个梯度字典，传输开销大 |
| **设备跨域问题** | 梯度可能在GPU上，显式传递需要在CPU/GPU间复制 |
| **代码冗余** | 每个参数都要手动传递，代码量爆炸 |
| **灵活性受限** | 无法在 `backward` 和 `step` 之间修改梯度（如梯度裁剪） |

**PyTorch的"暗度陈仓"设计优势：高效 + 省心！**

| 设计 | 优势 |
|------|------|
| **梯度附着在参数上** | 每个参数自己"背着"自己的梯度，不需要额外数据结构 |
| **原地操作** | 不复制张量，节省显存和计算 |
| **自动管理** | 计算图自动追踪，用户无需手动传递 |
| **灵活截胡** | 可以在 `backward` 和 `step` 之间修改梯度 |

```python
# ✅ 可以在背地交易中截胡：手动修改梯度
loss.backward()
with torch.no_grad():
    net.weight.grad *= 0.5  # 缩放梯度
optimizer.step()  # 读取修改后的梯度
```

**类比理解：** 
- **显式传参**：每次都要填写"梯度传递申请表"，经过层层审批（复制、传输），才能把梯度从loss办公室送到optimizer办公室——**效率低、不省心**
- **隐式传递（.grad）**：梯度自动"附着"在参数身上，走到哪儿跟到哪儿，optimizer直接从参数身上读取就行——**高效、自动、省心**

**一句话总结：PyTorch选择"暗度陈仓"是为了**高效、自动、省心**，让梯度自动附着在参数上，避免了显式传递的巨大开销和代码冗余。**

## 1.6 为什么优化的是 `net.parameters()` 而不是 `net`？

### 问题：能否写成 `optim.AdamW(net, lr=0.01)`？

**不能！** 优化器需要的是**可迭代的参数列表**，而不是模型对象本身。

```python
# ✅ 正确写法
optimizer = optim.AdamW(net.parameters(), lr=0.01)

# ❌ 错误写法（会报 TypeError）
# optimizer = optim.AdamW(net, lr=0.01)
```

### `net.parameters()` 到底是什么？

`net.parameters()` 返回的是**生成器（generator）**，包含了模型所有需要训练的参数：

```python
params = list(net.parameters())
print(f"参数数量：{len(params)}")  # 2 (weight 和 bias)
print(f"params[0] 是 weight: {params[0] is net.weight}")  # True
print(f"params[1] 是 bias: {params[1] is net.bias}")      # True
print(type(net.parameters()))  # <class 'generator'>
```

所以 `optim.AdamW(net.parameters(), lr=0.01)` 优化的是 **weight + bias** 两个张量。

### 为什么 PyTorch 这么设计？

虽然从"优化模型"的语义角度看，`optim.AdamW(net, lr=0.01)` 确实更直观，但 PyTorch 选择了**显式优于隐式**的设计哲学：

| 设计考量 | 说明 |
|---------|------|
| **一个模型可能有多个优化器** | 冻结部分层，只优化特定层时，需要精确控制 |
| **需要区分可训练和冻结参数** | `requires_grad=False` 的参数应被自动过滤 |
| **不同参数组需要不同超参数** | 层间差异化学习率是常见需求 |

```python
# 场景1：不同层不同学习率
optimizer = optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 1e-4},
    {'params': model.head.parameters(), 'lr': 1e-3}
])

# 场景2：只优化需要梯度的参数
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), 
    lr=0.01
)
```

### 社区常用简化写法
```python
# 封装成函数（非官方）
def create_optimizer(model, **kwargs):
    return optim.AdamW(model.parameters(), **kwargs)

optimizer = create_optimizer(net, lr=0.01)

# 或在模型类中封装
class MyModel(nn.Module):
    def configure_optimizers(self):
        return optim.AdamW(self.parameters(), lr=0.01)

optimizer = net.configure_optimizers()
```

**一句话总结：`optimizer` 要的是"参数列表"，不是"模型本身"，必须用 `.parameters()` 提取。**

## 过渡：优化器内部靠什么存储数据？

虽然日常训练中我们只需要调用 `optimizer.step()` 和 `optimizer.zero_grad()`，但理解优化器内部的数据结构有两个实际价值：

1. **断点续训**：保存和加载 `optimizer.state_dict()` 时，知道它为什么重要
2. **调试排查**：遇到问题时能理解 `param_groups` 和 `state` 的含义

### 用户视角 vs 内部实现

| 视角 | 需要关心的 | 不需要关心的 |
|------|----------|------------|
| **用户** | `optimizer.state_dict()` 保存/加载 | `param_groups` 的内部结构 |
| **用户** | 不同优化器需要保存不同状态 | `state` 里每个字段的具体含义 |
| **内部** | PyTorch 自动管理 | 用户无需手动操作 |

### 核心知识点

优化器内部有两套容器：

1. **`param_groups`**：`list[dict]`，存储配置（参数列表、超参数等），初始化时确定
2. **`state`**：`dict[Tensor, dict]`，存储运行时状态（动量缓存等），训练中动态累积

**不同优化器的 `state` 内容不同**：

| 优化器类型 | `state` 存储内容 | 原因 |
|-----------|-----------------|------|
| **无动量SGD** | `{}` 空字典 | 不需要历史信息 |
| **带动量SGD** | `{'momentum_buffer': tensor}` | 需要历史梯度平均 |
| **Adam/AdamW** | `{'step': 1, 'exp_avg': tensor, 'exp_avg_sq': tensor}` | 需要一阶/二阶动量 |

**实际意义**：这就是为什么断点续训时必须保存 `optimizer.state_dict()`——否则动量缓存丢失，训练会震荡。

> 💡 **用户不需要手动操作 `state` 和 `param_groups`，只需要知道：**
> 1. 保存检查点时，别忘了 `optimizer.state_dict()`
> 2. 加载检查点时，别忘了 `optimizer.load_state_dict()`

下面用打印实例来拆解这两个容器的具体结构（了解即可，日常训练无需手动操作）。

# 二、优化器底层核心存储结构

## 2.1 param_groups：参数组（list[dict]）

优化器初始化时接收一组/多组可训练参数，内部存储为参数组列表，每组独立配置超参。

### `param_groups` 存储什么？

`param_groups` 是一个 `list[dict]`，**每个字典存储该参数组的所有配置**：

| 配置项 | 说明 | 来源 |
|--------|------|------|
| `'params'` | 参数列表（必须的） | 用户传入 |
| `'lr'` | 学习率 | 用户指定 |
| `'weight_decay'` | 权重衰减系数 | 用户指定 |
| `'betas'` | Adam/AdamW 的动量系数 | 优化器默认值 |
| `'eps'` | 防止除零的小常数 | 优化器默认值 |
| `'amsgrad'` | 是否使用 AMSGrad 变体 | 优化器默认值 |
| ... | 其他优化器特定参数 | 优化器默认值 |

**关键点**：`param_groups` 存储的是**该参数组的所有超参数配置**，不限于 lr 和 weight_decay，而是优化器构造函数中**所有可配置的参数**。

### 不同优化器的 `param_groups` 内容不同

| 优化器 | `param_groups[0]` 的键 |
|--------|----------------------|
| **SGD** | `params`, `lr`, `momentum`, `weight_decay`, `dampening`, `nesterov`, `maximize` |
| **Adam** | `params`, `lr`, `betas`, `eps`, `weight_decay`, `amsgrad`, `maximize` |
| **AdamW** | `params`, `lr`, `betas`, `eps`, `weight_decay`, `amsgrad`, `maximize` |
| **RMSprop** | `params`, `lr`, `alpha`, `eps`, `weight_decay`, `momentum`, `centered` |

### 为什么只有 `param_groups[0]`？

```python
opt = optim.AdamW(net.parameters(), lr=0.001)
# 只传入了一组参数 → param_groups 长度为 1
print(len(opt.param_groups))  # 1
```

当你只传入 `net.parameters()` 时，PyTorch 会自动包装成一个参数组：

```python
# 你写的
opt = optim.AdamW(net.parameters(), lr=0.001)

# PyTorch 内部自动变成
opt = optim.AdamW([{'params': net.parameters(), 'lr': 0.001}])
#                  ↑ 列表里只有一个字典 → param_groups 长度为 1
```

**什么时候有多个参数组？** 手动传入多个字典时：

```python
opt = optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 1e-4},  # 组0
    {'params': model.head.parameters(), 'lr': 1e-3}       # 组1
])
print(len(opt.param_groups))  # 2
```

日常训练中 90% 的情况只有一个参数组，所以 `param_groups[0]` 是最常见的写法。

### 实例：打印 param_groups 的内容

下面的代码统一从 `param_groups` 读取所有内容，清晰展示它存储了什么。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
torch.manual_seed(42)

net = nn.Linear(2, 1)
opt = optim.AdamW(net.parameters(), lr=0.001, weight_decay=1e-4)

print("=" * 50)
print("📋 param_groups[0] 的内容")
print("=" * 50)

group = opt.param_groups[0]

# 遍历 param_groups[0] 的所有键值对
for key, value in group.items():
    if key == 'params':
        # params 是参数列表，需要特殊处理
        print(f"\n{key}:")
        # 建一个映射：参数id → 参数名（方便显示名字）
        id_to_name = {id(p): name for name, p in net.named_parameters()}
        for param in value:
            name = id_to_name.get(id(param), "unknown")
            print(f"    {name}: shape={param.shape}")
    else:
        # 其他配置项直接打印
        print(f"{key}: {value}")

print("\n" + "=" * 50)
print("📌 总结")
print("=" * 50)
print("  param_groups[0] 是一个字典，包含：")
print("    - 'params': 可训练参数列表（存的是参数对象，没有名字）")
print("    - 其他键: 超参数配置（lr, weight_decay, betas...）")
print("\n  💡 参数名字（weight/bias）是额外查表显示的，")
print("     并不存储在 param_groups 中。")

### 实例2：修改全局学习率（遍历param_groups）

In [ ]:
# 所有参数组学习率减半
for group in opt.param_groups:
    group['lr'] *= 0.5
print("修改后lr：", opt.param_groups[0]['lr'])

## 2.2 state：迭代状态缓存（dict）

### 基础定义

`optimizer.state` 是 `dict[Tensor, dict]`，存储每个参数的"运行时状态"（动量缓存等）。

**关键区分：`param_groups` vs `state` 都"有"参数，但存的完全不同**

| 位置 | 存的是什么 | 什么时候有 | 用途 |
|------|----------|----------|------|
| `param_groups['params']` | **参数对象本身**（weight/bias 张量数据） | 优化器初始化时就有 | 告诉优化器"要更新哪些参数" |
| `state` 的键 | **参数对象的引用**（作为字典的 key） | 第一次 `step()` 后生成 | 告诉优化器"每个参数的历史状态在哪" |

```
优化器内部：

param_groups:
    └── [0]
        ├── 'params': [weight对象, bias对象]    ← 参数本身
        ├── 'lr': 0.001
        └── ...

state:
    ├── weight对象 → {'step': 1, 'exp_avg': ..., 'exp_avg_sq': ...}  ← 参数的状态（缓存）
    └── bias对象   → {'step': 1, 'exp_avg': ..., 'exp_avg_sq': ...}  ← 参数的状态（缓存）
```

### 为什么 `state` 用参数对象做键？

因为优化器需要知道"**每个参数的**动量缓存是什么"，用参数对象作为 key 可以精确索引：

```python
# 伪代码：优化器 step() 内部
for param in self.param_groups[0]['params']:  # 遍历所有参数
    cache = self.state[param]                  # 用参数对象作为 key，找到它的缓存
    # 用 cache['exp_avg'] 更新这个参数
```

### 不同优化器存入state的缓存内容完全不同

| 优化器类型 | state存储字段 | 是否存储梯度平方 | 是否存储迭代步数t |
| ---- | ---- | ---- | ---- |
| 无动量SGD | 无，state恒为空 | ❌ 不存储 | ❌ 不存储 |
| 带动量SGD(momentum>0) | `momentum_buffer` | ❌ 不存储 | ❌ 不存储 |
| Adam / AdamW | `exp_avg`、`exp_avg_sq`、`step` | ✅ 存储 | ✅ 存储 |

#### 1）带动量SGD缓存明细
仅保存`momentum_buffer`：历史梯度的指数移动平均$m_{t-1}$，用于平滑梯度震荡，更新公式：$m_t = \beta \cdot m_{t-1} + (1-\beta)g_t$。

#### 2）Adam/AdamW缓存明细（工业最常用）
1. `exp_avg`：一阶动量，梯度指数移动平均，作用等同于带动量SGD的momentum_buffer；
2. `exp_avg_sq`：二阶动量，梯度平方的指数移动平均，用来自适应缩放每个参数的有效学习率；
3. `step`：标量迭代计数，记录该参数累计更新次数，用于前几轮迭代的动量偏差修正。

### 实例：完整展示 param_groups + state

下面的代码同时展示 `param_groups` 和 `state`，对比它们的区别。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
torch.manual_seed(42)

net = nn.Linear(2, 1)
opt = optim.AdamW(net.parameters(), lr=0.001, weight_decay=1e-4)

print("=" * 50)
print("📋 param_groups vs state 对比")
print("=" * 50)

# 1. param_groups（配置信息，初始化时就有）
print("\n【1】param_groups（配置信息，优化器初始化时就有）")
group = opt.param_groups[0]
id_to_name = {id(p): name for name, p in net.named_parameters()}
for key, value in group.items():
    if key == 'params':
        print("  params:")
        for param in value:
            name = id_to_name.get(id(param), "unknown")
            print(f"    {name}: shape={param.shape}")
    else:
        print(f"  {key}: {value}")

# 2. state（运行时状态，目前为空）
print("\n【2】state（运行时状态，目前为空）")
print(f"    {opt.state}")
print("    💡 state 为空，因为还没执行 step()")

# 3. 执行一步训练，生成 state
print("\n【3】执行 step() 后，state 被填充")
x = torch.rand(1, 2)
y = torch.rand(1, 1)
loss_fn = nn.MSELoss()

opt.zero_grad()
loss = loss_fn(net(x), y)
loss.backward()
opt.step()  # ← 第一次 step() 后，state 才生成

print("  step 后 state:")
for param, cache in opt.state.items():
    # 找到对应的参数名
    param_name = None
    for name, p in net.named_parameters():
        if p is param:
            param_name = name
            break
    print(f"    {param_name} (shape={param.shape}): {list(cache.keys())}")

print("\n" + "=" * 50)
print("📌 总结")
print("=" * 50)
print("  param_groups: 存储配置（超参数 + 参数列表），初始化时确定")
print("  state:        存储运行时状态（动量缓存），step() 后才生成")
print("\n💡 两者分工不同：param_groups 存'菜单'，state 存'操作台状态'")

### 关键注意：
state 缓存张量训练期间持续占用显存，训练结束后del optimizer可释放；断点续训必须同步保存优化器完整state。

# 三、优化器通用核心API详解
## 3.1 zero_grad() 梯度清零（错误+正确实例对比）
### 错误示范：忘记zero_grad，梯度累加，loss持续震荡

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
torch.manual_seed(42)
net = nn.Linear(2,1)
opt = optim.AdamW(net.parameters(), lr=0.01)
loss_fn = nn.MSELoss()
x,y = torch.rand(1,2), torch.rand(1,1)
for step in range(3):
    # 缺少 opt.zero_grad()，梯度持续累加
    loss = loss_fn(net(x), y)
    loss.backward()
    opt.step()
    print(f"Step {step+1} Loss: {loss.item():.4f}")
print("错误流程完成")

### 正确标准写法

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
torch.manual_seed(42)
net = nn.Linear(2,1)
opt = optim.AdamW(net.parameters(), lr=0.01)
loss_fn = nn.MSELoss()
x,y = torch.rand(1,2), torch.rand(1,1)
for step in range(3):
    opt.zero_grad()  # 每轮开头清零梯度
    loss = loss_fn(net(x), y)
    loss.backward()
    opt.step()
    print(f"Step {step+1} Loss: {loss.item():.4f}")
print("正确流程完成")

## 3.2 step() 参数更新
⚠️ 顺序硬性规则：zero_grad() → 前向+loss → backward() → step()

**`step()` 是优化器最核心的方法**：它负责读取当前 `param.grad` 或历史动量缓存，计算参数增量 Δθ，然后原地修改模型参数 θ。

```python
# 不同优化器在 step() 内部做的事情不同
SGD.step()：    θ = θ - lr * g_t
动量SGD.step()：先更新 m_t，再 θ = θ - lr * m_t
AdamW.step()：  先更新 m_t, v_t，再 θ = θ * (1-lr*λ) - lr * m_t/(√v_t+ε)
```

## 3.3 state_dict() / load_state_dict() 断点配套实例（完整代码放第8章节）
## 3.4 param_groups 动态调参实例（本章2.1小节已有可运行代码）

## 3.5 深入理解：优化器初始化时的参数传递方式
```python
# 三种等价写法
optimizer = optim.AdamW(net.parameters(), lr=0.01)  # 标准写法
optimizer = optim.AdamW([net.weight, net.bias], lr=0.01)  # 显式传参
optimizer = optim.AdamW([{'params': net.parameters()}], lr=0.01)  # 参数组写法

# 不能写成：
# optimizer = optim.AdamW(net, lr=0.01)  # ❌ TypeError
```

**关键点**：
- `net.parameters()` 返回生成器，包含 weight 和 bias
- 优化器内部会自动将生成器包装成 `[{'params': 生成器}]`
- 多参数组时必须显式写 `'params'` 键

# 四、主流优化器完整原理与公式对比

## 统一框架：所有优化器 = 梯度 × 处理策略

| 优化器 | 使用的数据 | step() 内部处理策略 | step() 后参数变化 |
|--------|-----------|-------------------|------------------|
| **SGD（无动量）** | 当前步梯度 `g_t` | 直接用：`Δθ = lr × g_t` | `θ = θ - Δθ` |
| **SGD（带动量）** | 历史梯度指数平均 `m_t` | 先更新 `m_t = β*m_{t-1} + g_t`，再用 `m_t` | `θ = θ - lr × m_t` |
| **Adam/AdamW** | 一阶动量 `m_t` + 二阶动量 `v_t` | 先更新 m_t、v_t，修正偏差，计算自适应步长 | 每个参数独立步长更新 |
| **RMSprop** | 二阶动量 `v_t` | 缩放梯度，防止震荡 | 自适应步长更新 |
| **Adagrad** | 二阶动量累积 | 稀疏特征大步长，密集特征小步长 | 自适应步长更新 |

## 符号统一约定
- $\theta_t$：第t步模型参数
- $g_t = \nabla \mathcal{L}(\theta_t)$：原始损失梯度（所有优化器共享同一个数据来源）
- $\eta$：基础学习率 lr
- $\lambda$：weight_decay 权重衰减系数
- $\beta_1$：一阶动量系数，$\beta_2$：二阶动量系数
- $m_t$：一阶动量均值，$v_t$：二阶动量平方均值
- $\hat{m}_t,\hat{v}_t$：动量偏差修正项
- $\epsilon$：防止分母为0的极小常数 $10^{-8}$

## 4.1 SGD 带动量 + L2正则
### 标准更新公式
$$
\begin{aligned}
m_t &= \beta_1 m_{t-1} + g_t \\
\theta_{t+1} &= \theta_t - \eta \cdot \big(m_t + \lambda \theta_t\big)
\end{aligned}
$$
### step() 内部执行步骤
1. 读取当前梯度 `g_t`
2. 更新动量缓存：`m_t = β₁·m_{t-1} + g_t`
3. 计算参数增量：`Δθ = η·(m_t + λ·θ_t)`
4. 原地更新参数：`θ_t = θ_t - Δθ`

### 特点
1. 无自适应学习率，全局固定lr；
2. 原生L2正则，衰减直接叠加梯度，不会被自适应缩放扭曲；
3. 泛化性能强，小数据集首选，收敛速度慢、调参难度高。
4. **梯度使用方式**：用历史梯度指数平均 `m_t` 替代当前梯度 `g_t`，平滑震荡

## 4.2 原始 Adam
### 完整公式
$$
\begin{aligned}
m_t &= \beta_1 m_{t-1} + (1-\beta_1) g_t \\
v_t &= \beta_2 v_{t-1} + (1-\beta_2) g_t^2 \\
\hat{m}_t &= \frac{m_t}{1-\beta_1^t},\quad \hat{v}_t = \frac{v_t}{1-\beta_2^t} \\
\theta_{t+1} &= \theta_t - \eta \cdot \left(
\frac{\hat{m}_t}{\sqrt{\hat{v}_t}+\epsilon}
+ \lambda \cdot \theta_t
\right)
\end{aligned}
$$
### step() 内部执行步骤
1. 读取当前梯度 `g_t`
2. 更新一阶动量：`m_t = β₁·m_{t-1} + (1-β₁)·g_t`
3. 更新二阶动量：`v_t = β₂·v_{t-1} + (1-β₂)·g_t²`
4. 偏差修正：`m̂_t = m_t/(1-β₁ᵗ)`, `v̂_t = v_t/(1-β₂ᵗ)`
5. 计算参数增量并更新

### 致命缺陷
权重惩罚项混入梯度，参与一、二阶动量累积；自适应分母缩放衰减力度，weight_decay正则效果大幅削弱，容易过拟合。
### 梯度使用方式
同时维护一阶动量 `m_t`（方向）和二阶动量 `v_t`（步长缩放），每个参数获得独立自适应学习率

## 4.3 AdamW（工业标准推荐优化器）
### 核心改动
权重衰减从梯度剥离，单独后置作用于原始参数，不参与动量、二阶方差的累积计算。
### 完整公式
$$
\begin{aligned}
m_t &= \beta_1 m_{t-1} + (1-\beta_1) g_t \\
v_t &= \beta_2 v_{t-1} + (1-\beta_2) g_t^2 \\
\hat{m}_t &= \frac{m_t}{1-\beta_1^t},\quad \hat{v}_t = \frac{v_t}{1-\beta_2^t} \\
\theta_{t+1} &= \theta_t (1 - \eta \lambda) - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t}+\epsilon}
\end{aligned}
$$
### step() 内部执行步骤
1. 读取当前梯度 `g_t`
2. 更新一阶动量：`m_t = β₁·m_{t-1} + (1-β₁)·g_t`
3. 更新二阶动量：`v_t = β₂·v_{t-1} + (1-β₂)·g_t²`（不包含权重衰减）
4. 偏差修正：`m̂_t`, `v̂_t`
5. 权重衰减独立后置：`θ_t = θ_t·(1 - η·λ)`
6. 梯度更新：`θ_t = θ_t - η·m̂_t/(√v̂_t + ε)`

### 优势
1. 动量仅使用原始梯度，不受权重衰减干扰；
2. 衰减项独立后置，参数收缩强度稳定可控；
3. 收敛速度快+正则效果可靠，绝大多数深度学习任务默认推荐。
### 梯度使用方式
与Adam相同的一二阶动量自适应，但权重衰减独立后置，不污染动量计算

## 4.4 三优化器一行对照实例

In [ ]:
import torch.nn as nn
import torch.optim as optim
torch.manual_seed(42)
model = nn.Linear(10,2)
lr, wd = 1e-3, 1e-4
opt_sgd = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
opt_adam = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
opt_adamw = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
print("SGD优化器:", opt_sgd)
print("Adam优化器:", opt_adam)
print("AdamW优化器:", opt_adamw)
print("\n💡 三种优化器都从 param.grad 读取梯度，但 step() 内部算法不同")

# 五、torch.optim.lr_scheduler 学习率调度器

## 5.0 调度器本质：一句话先给结论

**`torch.optim.lr_scheduler` 就是一套"按规则自动调整学习率"的工具**，它负责在训练过程中，根据你设定的策略（比如每过几轮、或者损失不再下降时），动态地增大或减小优化器里的学习率。

## 5.1 为什么要调度学习率？

固定学习率有两大痛点：
- **太大**：模型在最优解附近来回震荡，就是不收敛。
- **太小**：训练极慢，还容易陷在局部最优里出不来。

好的策略是 **"先快后慢"**：训练初期用大学习率快速找到"好区域"，后期用小学习率慢慢微调，找到最优点。调度器干的就是这个"动态调整"的活。

## 5.2 调度器本质

不修改优化器更新数学逻辑，仅周期性修改 `optimizer.param_groups` 内部的 `lr` 值。

## 5.3 核心使用流程（最容易犯错的地方）

这是新手最容易踩坑的一点，记住这个**标准三步走**：

```python
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler

# 1. 先定义优化器
optimizer = optim.AdamW(model.parameters(), lr=0.1)

# 2. 再定义调度器（把优化器传进去）
scheduler = lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)

# 3. 训练循环中，每轮（或每步）后调用 step()
for epoch in range(100):
    train(...)  # 正常训练
    scheduler.step()  # 更新学习率（注意：是每个epoch结束后调用）
```

**关键点**：`scheduler.step()` 必须在 `optimizer.step()` **之后**调用，否则学习率变化会提前一个批次，导致训练异常。

## 5.4 常用调度器类型及适用场景

| 类型 | 策略 | 适用场景 |
|------|------|----------|
| **StepLR** | 每隔固定轮数，学习率乘上一个系数（如每30轮乘以0.1） | 经典基线模型，你大概知道什么时候该衰减 |
| **MultiStepLR** | 在指定的几个轮次（如[30, 60, 80]）衰减 | 比StepLR更灵活，常用在ResNet等 |
| **ExponentialLR** | 每轮都指数衰减（`lr = lr * gamma^epoch`） | 需要平滑连续衰减时 |
| **ReduceLROnPlateau** | 监控验证集指标，指标停滞就自动衰减 | **实战最推荐**，不用猜什么时候衰减，让它自己判断 |
| **CosineAnnealingLR** | 余弦退火，学习率按余弦周期变化 | 常用于大模型和SGDR，能跳出局部最优 |

## 5.5 通用调用规则

- 绝大多数常规训练场景：每个epoch循环末尾执行 `scheduler.step()`
- 特例：`CosineAnnealingWarmRestarts` 科研场景可按batch更新
- 特例：`ReduceLROnPlateau` 需要传入验证损失 `scheduler.step(val_loss)`

## 实例1：StepLR 固定间隔衰减

In [ ]:
import torch.nn as nn
import torch.optim as optim
torch.manual_seed(42)
model = nn.Linear(10,2)
opt = optim.AdamW(model.parameters(), lr=0.01)
# 每10个epoch，学习率乘以0.5
scheduler_step = optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5)

## 实例2：MultiStepLR 指定节点衰减（更灵活）

In [ ]:
# 在第30、60、80个epoch时，学习率乘以0.1
scheduler_multi = optim.lr_scheduler.MultiStepLR(opt, milestones=[30, 60, 80], gamma=0.1)

## 实例3：ExponentialLR 指数衰减

In [ ]:
# 每轮衰减为原来的 0.95 倍
scheduler_exp = optim.lr_scheduler.ExponentialLR(opt, gamma=0.95)

## 实例4：CosineAnnealingLR 余弦退火

In [ ]:
# 总训练100epoch，lr平滑余弦下降
scheduler_cosine = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100)

## 实例5：ReduceLROnPlateau 验证集自适应衰减（实战最推荐）

⚠️ **注意**：这种调度器的 `step()` 需要传入验证集损失，而不是空调用！

In [ ]:
# 验证loss连续5轮不下降，lr乘0.7
scheduler_plateau = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.7, mode='min')

## 实例6：CosineAnnealingWarmRestarts 带重启余弦退火

In [ ]:
# 每20轮重启一次余弦周期
scheduler_restart = optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=20)

## 5.6 进阶用法：不同层不同学习率 + 调度器

如果你的模型里，不同层想用不同的学习率（比如骨干网络用1e-5，分类头用1e-3），调度器会按比例调整每组自己的lr：

In [ ]:
import torch.nn as nn
from torch.optim import AdamW
torch.manual_seed(42)

# 构建复合模型（有 backbone 和 head）
class Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 64, kernel_size=3)

class ClassHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(64, 10)

# 用 Sequential 组合
model = nn.Sequential(Backbone(), ClassHead())

# 这样就有 backbone 和 head 了
optimizer = AdamW([
    {'params': model[0].parameters(), 'lr': 1e-5},  # backbone
    {'params': model[1].parameters(), 'lr': 1e-3}   # head
], lr=1e-4)

scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

## 5.7 查看当前学习率

调度器修改的是优化器里的 `param_groups`，你可以随时打印查看：

In [ ]:
print(optimizer.param_groups[0]['lr'])  # 查看当前学习率

## 5.8 标准训练搭配完整示例

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
torch.manual_seed(42)
model = nn.Linear(10,2)
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
# 模拟训练循环
for epoch in range(10):
    # 模拟训练步骤
    scheduler.step()  # epoch结束更新学习率
    print(f"Epoch {epoch+1}, 当前lr: {optimizer.param_groups[0]['lr']:.6f}")

## 5.9 💡 实践经验建议

如果你刚开始做项目：
1. 先用 **`ReduceLROnPlateau`**（模式设为 `'min'`，耐心值 `patience=10`），让它自动帮你调。
2. 等跑通基线后，再换成 **`CosineAnnealingLR`** 或 **`OneCycleLR`** 去刷精度。

如果你想理解论文里的写法，那八成是 `StepLR` 或 `MultiStepLR`，因为简单、可复现。

# 六、高级用法：分层差异化参数训练
## 业务场景：CNN主干预训练权重小学习率，分类头大学习率

In [ ]:
import torch.nn as nn
from torch.optim import AdamW
torch.manual_seed(42)

# 模拟骨干网络 + 分类头
class Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 64, kernel_size=3)

class ClassHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(64, 10)

model = nn.Sequential(Backbone(), ClassHead())

# 拆分两组参数，独立设置lr、weight_decay
optimizer = AdamW([
    {
        "params": model[0].parameters(),
        "lr": 1e-4,        # 主干小学习率
        "weight_decay": 1e-5
    },
    {
        "params": model[1].parameters(),
        "lr": 1e-3,        # 分类头大学习率
        "weight_decay": 1e-4
    }
])

# 分别打印两组学习率
print("主干lr:", optimizer.param_groups[0]['lr'])
print("分类头lr:", optimizer.param_groups[1]['lr'])
print("\n💡 param_groups 支持每组独立超参，实现差异化训练策略")

## 拓展实例：冻结主干网络（requires_grad=False自动过滤，不参与更新）

In [ ]:
# 冻结主干所有参数
for p in model[0].parameters():
    p.requires_grad = False
# 创建优化器时自动过滤requires_grad=False参数，仅保留分类头
opt_freeze = AdamW(model.parameters(), lr=1e-3)
print("冻结后可训练参数数量:", len(opt_freeze.param_groups[0]['params']))
print("\n💡 requires_grad=False 的参数不会出现在优化器的 param_groups 中")

# 七、完整标准训练流程代码模板

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
torch.manual_seed(42)

# 1. 构建模型、损失函数、模拟数据集
model = nn.Sequential(nn.Linear(16, 64), nn.ReLU(), nn.Linear(64, 2))
loss_fn = nn.CrossEntropyLoss()

x_data = torch.randn(1000, 16)
y_data = torch.randint(0, 2, (1000,))
dataloader = DataLoader(TensorDataset(x_data, y_data), batch_size=32, shuffle=True)

# 2. 初始化优化器 + 学习率调度器
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.7)

# 3. 迭代训练循环
total_epoch = 10
for epoch in range(total_epoch):
    total_loss = 0.0
    model.train()
    for batch_x, batch_y in dataloader:
        # 步骤1：清空梯度（必写）
        optimizer.zero_grad()
        # 步骤2：前向传播
        pred = model(batch_x)
        loss = loss_fn(pred, batch_y)
        # 步骤3：反向传播求梯度（生成 .grad）
        loss.backward()
        # 步骤4：优化器更新参数（读取 .grad 并应用策略）
        optimizer.step()

        total_loss += loss.item()
    # epoch结束更新学习率
    scheduler.step()
    avg_loss = total_loss/len(dataloader)
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1:2d} | Avg Loss: {avg_loss:.4f} | LR: {current_lr:.6f}")

# 八、断点保存/加载优化器完整实例（state_dict实操）
## 8.1 保存模型+优化器完整状态

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(16,64), nn.ReLU(), nn.Linear(64,2))
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
epoch = 10
# 训练中途保存断点
torch.save({
    "model_weight": model.state_dict(),
    "optim_state": optimizer.state_dict(),  # 必须保存优化器动量缓存
    "epoch": epoch,
    "lr": optimizer.param_groups[0]['lr']
}, "train_checkpoint.pth")
print("断点文件保存完成")
print("💡 optimizer.state_dict() 包含了所有动量缓存（exp_avg, exp_avg_sq, step）")

## 8.2 加载断点续训完整代码（兼容CPU/GPU）

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
torch.manual_seed(42)
# 1. 重建模型、优化器（结构必须和保存时完全一致）
new_model = nn.Sequential(nn.Linear(16,64), nn.ReLU(), nn.Linear(64,2))
new_opt = optim.AdamW(new_model.parameters(), lr=1e-3, weight_decay=1e-4)

# 2. 读取断点文件，map_location兼容GPU保存、CPU加载
checkpoint = torch.load("train_checkpoint.pth", map_location=torch.device("cpu"))

# 3. 加载权重 + 优化器动量缓存
new_model.load_state_dict(checkpoint["model_weight"])
new_opt.load_state_dict(checkpoint["optim_state"])  # ✅ 修复：使用 new_opt 而非 optimizer

start_epoch = checkpoint["epoch"] + 1
print(f"从第{start_epoch}轮继续训练")
print("💡 恢复优化器state后，动量缓存完整，训练可平稳继续")

**踩坑提醒：只保存模型不保存opt.state_dict()，AdamW动量缓存丢失，训练会震荡发散。**

# 九、高频踩坑与避坑指南 ⚠️
## 9.1 梯度相关错误
1. 遗漏 `zero_grad()`：梯度持续累加，loss震荡不收敛；第三章3.1有对照代码
2. `step()` 写在 `backward()` 之前：梯度为空，参数完全不更新

### 拓展答疑：梯度默认累加，为何PyTorch要这么设计？
#### 核心设计目的：首要支持梯度累积（小显卡模拟大batch，多批数据攒梯度后一次性更新权重）
1. 底层计算规则：每次`loss.backward()`执行原地加法 `w.grad += 新梯度`，不会直接覆盖原有梯度；
2. 核心场景1：梯度累积（显存不足等效大batch训练）
显卡放不下大批次数据时，连续多轮backward叠加梯度，累积N个小batch后仅执行一次step更新权重，代码模板：


In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW  # ✅ 修复：添加导入
torch.manual_seed(42)
accum_steps = 4
model = nn.Linear(8,2)
loss_fn = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=1e-3)
dataloader = [(torch.rand(2,8), torch.randint(0,2,(2,))) for _ in range(12)]

for idx, (x,y) in enumerate(dataloader):
    pred = model(x)
    loss = loss_fn(pred, y) / accum_steps
    loss.backward() # 梯度自动累加，不清零
    if (idx+1) % accum_steps == 0:
        optimizer.step()
        optimizer.zero_grad()
print("梯度累积训练完成")
print("💡 梯度累加设计让 PyTorch 支持小显存模拟大batch训练")

3. 核心场景2：多任务联合损失叠加
分类、分割、检测多任务共用一套参数，多次backward自动叠加多分支梯度，无需手动拼接梯度张量；
4. 核心场景3：高级算法高阶梯度计算
WGAN-GP梯度惩罚、海森矩阵、二阶梯度等研究类算法，依赖梯度可累加的底层特性；

#### 为什么不默认自动清零梯度？
若backward自动覆盖/清空grad，梯度累积、多任务叠加、高阶梯度全部无法原生实现；
PyTorch底层遵循「最小限制基础能力」设计思路，常规训练手动写zero_grad()即可，进阶场景直接复用累加特性，不用新增额外API。

## 9.2 权重衰减误区
❌ 错误：Adam + weight_decay = 正则失效
✅ 正确：AdamW 替代

## 9.3 调度器调用错误

### ❌ 错误1：在batch循环内部执行 scheduler.step()
```python
for epoch in range(epochs):
    for batch in dataloader:
        train_step(batch)
        scheduler.step()  # ❌ 每batch都衰减，lr下降过快
```

### ✅ 正确：每个epoch结束执行一次
```python
for epoch in range(epochs):
    for batch in dataloader:
        train_step(batch)
    scheduler.step()  # ✅ 每epoch结束衰减一次
```

### ❌ 错误2：scheduler.step() 在 optimizer.step() 之前调用
```python
for batch in dataloader:
    scheduler.step()  # ❌ 先改lr
    loss.backward()
    optimizer.step()  # 用改后的lr更新，顺序错乱
```

### ✅ 正确：optimizer.step() 之后调用
```python
for batch in dataloader:
    loss.backward()
    optimizer.step()  # ✅ 先用当前lr更新
    scheduler.step()  # ✅ 再调整下一轮用的lr
```

> ⚠️ **注意**：上述示例中的 `scheduler.step()` 在 batch 循环内调用仅适用于 `CosineAnnealingWarmRestarts` 等支持 per-batch 更新的调度器。对于 `StepLR`、`MultiStepLR`、`ReduceLROnPlateau` 等，应在 **epoch 循环结束后**调用。

### ❌ 错误3：ReduceLROnPlateau 不传验证损失
```python
scheduler_plateau.step()  # ❌ 缺少 val_loss 参数，会报错
```

### ✅ 正确：传入验证集损失
```python
scheduler_plateau.step(val_loss)  # ✅ 监控验证损失
```

## 9.4 显存/断点问题
1. 冻结层 `requires_grad=False` 传入优化器会被自动过滤，不会占用梯度显存；
2. 断点续训仅保存模型权重、不保存 `optimizer.state_dict()`：动量缓存丢失，训练震荡发散；

## 9.5 梯度裁剪配套实例（backward和step中间使用）

In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW  # ✅ 修复：添加导入
torch.manual_seed(42)
net = nn.Linear(2,1)
optimizer = AdamW(net.parameters(), lr=0.01)
loss_fn = nn.MSELoss()
x,y = torch.rand(1,2), torch.rand(1,1)

optimizer.zero_grad()
loss_fn(net(x), y).backward()
# 梯度裁剪，防止梯度爆炸
torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
optimizer.step()
print("梯度裁剪流程执行完成")
print("💡 梯度裁剪在 backward() 和 step() 之间执行，直接修改 .grad 值")

## 9.6 优化器初始化常见错误

### ❌ 错误：直接传入模型对象
```python
# 这样写会报错！
optimizer = optim.AdamW(net, lr=0.01)  # TypeError: 'Linear' object is not iterable
```

### ✅ 正确：传入模型参数
```python
optimizer = optim.AdamW(net.parameters(), lr=0.01)
```

### 为什么必须是 `net.parameters()`？
| 原因 | 说明 |
|------|------|
| **优化器需要参数列表** | `net.parameters()` 返回一个生成器，产出所有 `requires_grad=True` 的参数 |
| **支持参数分组** | 不同层可以用不同学习率，通过 `param_groups` 实现 |
| **自动过滤冻结参数** | `requires_grad=False` 的参数会被自动跳过 |

### 验证 `net.parameters()` 的内容
```python
params = list(net.parameters())
print(f"参数数量: {len(params)}")  # 2 (weight 和 bias)
print(f"weight: {params[0].shape}, bias: {params[1].shape}")
```

### 🎭 回顾："明修栈道，暗度陈仓"
优化器和loss看似独立，实则通过 `param.grad` 完成秘密交易：
- `loss.backward()` → 计算梯度，写入 `param.grad`（暗地传递情报）
- `optimizer.step()` → 读取 `param.grad`，更新参数（接收情报执行行动）
- 之所以不设计成显式传参，是因为**高效、自动、省心**

### 🔄 对比：显式传参 vs 隐式传递

| 对比维度 | 显式传参（假想设计） | 隐式传递（PyTorch实际） |
|---------|-------------------|---------------------|
| **代码形式** | `grad = loss.backward(); optimizer.step(grad)` | `loss.backward(); optimizer.step()` |
| **参数传递** | 需要显式传递梯度字典 | 梯度附着在参数上，自动传递 |
| **内存开销** | 需要复制/传输梯度 | 原地操作，无额外开销 |
| **灵活性** | 难以在中间插入操作 | 可以在 backward 和 step 之间任意操作 |
| **设备兼容** | 需要在 CPU/GPU 间传输 | 梯度自动在正确设备上 |
| **代码简洁** | 冗余，每个参数都要传递 | 简洁，一行 `step()` 搞定 |

**一句话总结：PyTorch 的设计哲学是"显式API背后，隐藏着高效的隐式梯度传递机制"——让你省心、高效、灵活地训练模型。**

### 💡 关于优化器内部存储结构的总结

| 用户需要做的 | 用户不需要做的 |
|------------|--------------|
| ✅ 保存检查点时包含 `optimizer.state_dict()` | ❌ 手动操作 `optimizer.param_groups` |
| ✅ 加载检查点时调用 `optimizer.load_state_dict()` | ❌ 手动操作 `optimizer.state` |
| ✅ 理解为什么需要保存 `state_dict`（动量缓存） | ❌ 关心 `state` 里每个字段的具体含义 |

**核心要点**：优化器内部自动管理 `param_groups` 和 `state`，用户只需要在断点续训时正确保存/加载 `optimizer.state_dict()`。

### 📌 快速回顾：`param_groups` 和 `state` 的区别

| 对比维度 | `param_groups` | `state` |
|---------|---------------|---------|
| **类型** | `list[dict]` | `dict[Tensor, dict]` |
| **存储内容** | 配置：超参数 + 参数列表 | 运行时状态：动量缓存等 |
| **何时生成** | 优化器初始化时 | 第一次 `step()` 后 |
| **用户操作** | 一般不需要手动修改 | 不需要手动操作 |
| **保存/加载** | 包含在 `state_dict()` 中 | 包含在 `state_dict()` 中 |

# 十、附录：SGD每次迭代使用的样本数量（batch_size详解）

## 🎯 直接回答你的问题

**SGD每次训练迭代（即每次 `optimizer.step()`）使用的样本数量 = batch_size**

**这个值是可以设置的！** 通过 `DataLoader` 的 `batch_size` 参数控制。

---

## 📊 代码演示：如何设置 batch_size

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

print("=" * 60)
print("📦 示例1：设置 batch_size（每次迭代使用的样本数）")
print("=" * 60)

# 准备数据集：1000条样本
torch.manual_seed(42)
x_data = torch.randn(1000, 16)   # 1000条样本，每条16维
y_data = torch.randint(0, 2, (1000,))  # 1000个标签
dataset = TensorDataset(x_data, y_data)

# ✅ batch_size 可以自由设置！
# 常见取值：32, 64, 128, 256（取决于显存大小）
batch_size = 64  # 👈 这就是你问的"每次迭代使用的样本数量"

dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"📌 数据集总样本数: {len(dataset)}")
print(f"📌 batch_size = {batch_size}")
print(f"📌 每个epoch的迭代步数: {len(dataloader)}")
print(f"   （因为 {len(dataset)} ÷ {batch_size} = {len(dataloader)} 步）")

# 验证：取一个batch看看
for batch_idx, (x, y) in enumerate(dataloader):
    print(f"\n   Batch {batch_idx + 1}: x.shape = {x.shape}, y.shape = {y.shape}")
    print(f"   这个batch包含 {x.shape[0]} 条样本 → 这就是一次 step() 用的数据量")
    if batch_idx >= 2:  # 只展示前3个batch
        break

print("\n" + "=" * 60)
print("💡 结论：batch_size 就是每次迭代使用的样本数量，可以自由设置！")
print("   改大 → 每次 step 看更多数据，但更耗显存")
print("   改小 → 每次 step 看更少数据，更省显存，但梯度噪声更大")

## 🔄 不同 batch_size 对训练的影响

下面用实际训练对比不同 batch_size 的效果：

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

print("=" * 60)
print("🏋️ 示例2：不同 batch_size 的训练对比")
print("=" * 60)

torch.manual_seed(42)

# 准备数据
x_data = torch.randn(500, 16)
y_data = torch.randint(0, 2, (500,))
dataset = TensorDataset(x_data, y_data)

# 定义模型
def create_model():
    return nn.Sequential(
        nn.Linear(16, 32),
        nn.ReLU(),
        nn.Linear(32, 2)
    )
loss_fn = nn.CrossEntropyLoss()

# 测试三种 batch_size
batch_sizes = [16, 64, 256]

print(f"{'batch_size':>10} | {'每步样本数':>12} | {'每轮步数':>10} | {'显存占用':>10}")
print("-" * 60)

for bs in batch_sizes:
    # 重新初始化模型
    model = create_model()
    optimizer = optim.SGD(model.parameters(), lr=0.01)
    dataloader = DataLoader(dataset, batch_size=bs, shuffle=True)
    
    steps_per_epoch = len(dataloader)
    
    # 估算显存占用（粗略）
    # 每个batch占用 ≈ batch_size × 16 × 4字节（float32）
    mem_approx = bs * 16 * 4 / 1024  # KB
    
    print(f"{bs:>10} | {bs:>12} | {steps_per_epoch:>10} | {mem_approx:>9.1f} KB")
    
    # 训练2个epoch看看loss变化
    losses = []
    for epoch in range(2):
        total_loss = 0.0
        for batch_x, batch_y in dataloader:
            optimizer.zero_grad()
            pred = model(batch_x)
            loss = loss_fn(pred, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / steps_per_epoch
        losses.append(avg_loss)
    
    print(f"           | 第1轮loss: {losses[0]:.4f} | 第2轮loss: {losses[1]:.4f}")
    print("-" * 60)

print("\n💡 观察：")
print("   • batch_size 小（16）：每轮步数多，梯度噪声大，但收敛可能更快")
print("   • batch_size 大（256）：每轮步数少，梯度更稳定，但更耗显存")
print("   • batch_size 就是每次 step() 看到的样本数，可以直接调整！")

## 🎯 一句话总结

| 问题 | 答案 |
|------|------|
| SGD每次迭代用多少样本？ | **batch_size** 条 |
| 可以设置吗？ | **✅ 可以！** |
| 怎么设置？ | `DataLoader(dataset, batch_size=你想要的数字)` |
| 常见取值？ | 32, 64, 128, 256（取决于显存） |
| 改大有什么影响？ | 更耗显存，梯度更稳定 |
| 改小有什么影响？ | 更省显存，梯度噪声更大 |